In [5]:
from langchain_community.document_loaders import PDFPlumberLoader

c:\Users\Yashita\Desktop\film_lens_ai\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
loader = PDFPlumberLoader(r"C:\Users\Yashita\Desktop\film_lens_ai\data\rag_data\ncert_film_making_chapter.pdf")

In [13]:
docs = loader.load()

In [14]:
len(docs)

16

In [15]:
docs

[Document(metadata={'source': 'C:\\Users\\Yashita\\Desktop\\film_lens_ai\\data\\rag_data\\ncert_film_making_chapter.pdf', 'file_path': 'C:\\Users\\Yashita\\Desktop\\film_lens_ai\\data\\rag_data\\ncert_film_making_chapter.pdf', 'page': 0, 'total_pages': 16, 'Author': 'NCERT', 'CreationDate': 'D:20171223141408Z', 'Creator': 'PageMaker 7.0', 'ModDate': "D:20250509134230+05'30'", 'Producer': 'GPL Ghostscript 8.15', 'Title': 'chap-03.pmd'}, page_content='149/FILM-MAKING\n33333\nFFFFFiiiiilllllmmmmm-----mmmmmaaaaakkkkkiiiiinnnnnggggg\nIngmar Bergman is a well known Swedish director of\nfilms noted for their starkness, their subtle use of black\nand white and ‘shades’ of those extremes, the ambiguity\nof their content, and a certain brooding presence that\nseems to pervade them all. The list of Bergman films is\nlong; his best known include The Seventh Seal (1957),\nWild Strawberries (1958), The Virgin Spring (1960),\nThe Silence (1963), Persona (1967), The Passion of\nAnna (1970), and Cries 

In [28]:
from langchain_text_splitters import TokenTextSplitter

In [39]:
import tiktoken
print("tiktoken version:", tiktoken.__version__)

tiktoken version: 0.12.0


In [40]:
import tiktoken

def chunk_text_tokens(
    text: str,
    chunk_size: int = 200,
    overlap: int = 50,
    model: str = "gpt-4o-mini"
):
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        start += chunk_size - overlap

    return chunks


In [41]:
all_chunks = []

for doc in docs:   # docs = your loaded PDF texts
    text_chunks = chunk_text_tokens(doc.page_content)
    for chunk in text_chunks:
        all_chunks.append({
            "text": chunk,
            "metadata": doc.metadata
        })

print("Total chunks:", len(all_chunks))


Total chunks: 56


In [43]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-V2")

texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, convert_to_numpy=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [46]:
faiss.write_index(index,"faiss.index")

In [47]:
import pickle

with open("faiss_metadata.pkl","wb") as f:
    pickle.dump(all_chunks,f)